# 움직임 시험 1 — 블로킹(움직임만 잡은 밑그림) → LoRA 그림→그림(img2img)으로 장마다 다시 그리기 (Kaggle T4, 학습 없음)
목적: 움직임이 정해진 밑그림을 컬러 LoRA가 나카무라 화풍으로 장마다 다시 그릴 수 있는지, 장 사이 깜빡임이 얼마나인지 본다.
- 입력 A `num`: 숫자로 만든 블로킹 15장 — 나카무라 폭발 실측 타이밍(길이 15장, 정점 6장째, 화면 22.5%, 1·2·3장 걸이, 4번 갈라짐, 파편 감속; tools/fx_blocking.py). 둥근 덩어리라 실루엣이 단순.
- 입력 B `clip`: 나카무라 폭발 장면(c0348, 건물 뒤에서 부푸는 폭발) 12장에서 실루엣과 움직임만 남기고 화풍을 지운 블로킹(tools/fx_blocking_clip.py). 배경은 A와 같은 밤 도시.
- LoRA: 컬러 3차 3바퀴(grey smoke 태그·마스크 손실 학습). 부정어는 기본(부정어 grey smoke는 보라·청록 덩어리가 끼어 이번엔 안 씀).
- 강도(원래 그림을 얼마나 바꿀지) 0.4 · 0.55 · 0.7, 모든 장 같은 시드(1234)·같은 글.
- 설정: **GPU T4 x2**, **Internet On** → **Save Version → Save & Run All (Commit)**. 약 40–50분.
- 끝나면 Output의 `motion1_result.zip`(나란히 영상 `motion1.mp4` + 장별 그림)을 Drive 공유 링크로 알려 준다.


In [ ]:
# 1) 학습 환경: 서버에서 불러오기까지 확인한 조합을 별도 환경(/tmp/env)에 그대로 깐다(Kaggle 기본 패키지와 섞이지 않게)
!pip -q install uv
!uv venv -q /tmp/env --python 3.10
!uv pip install -q --python /tmp/env/bin/python torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
!git clone -q --depth 1 --branch v0.9.1 https://github.com/kohya-ss/sd-scripts.git /tmp/sd-scripts
!cd /tmp/sd-scripts && uv pip install -q --python /tmp/env/bin/python -r requirements.txt
!uv pip install -q --python /tmp/env/bin/python lycoris-lora==3.1.1 "numpy==1.26.4"
!cd /tmp/sd-scripts && /tmp/env/bin/python -c "import torch, library, lycoris, bitsandbytes, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 2) 기본 모델(Hugging Face)과 컬러 LoRA 3차 3바퀴(Drive 공유 링크, 2026-10-26 판정: 3차 중 가장 좋음)
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!mkdir -p /tmp/out && curl -sSL -o /tmp/out/nkmr_fx_color3-000003.safetensors "https://drive.usercontent.google.com/download?id=1ht_UqE8JkJTUX-eO5lNIqvhyFRcprUWJ&export=download&confirm=t"
!ls -la /tmp/ilxl.safetensors /tmp/out/


In [ ]:
# 3) 블로킹 두 가지(Drive)와 강도별 그림→그림
import os, glob
for name, fid in [("num", "1lLCSZbiHE0QUO6xSw_uIeWiyUoNjlgDx"), ("clip", "1jITc3rXRduBnoW3C05I2XLmJ_7hEdtI6")]:
    os.system(f'curl -sSL -o /tmp/{name}.zip "https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t"')
    os.makedirs(f"/tmp/blk_{name}", exist_ok=True)
    os.system(f"cd /tmp/blk_{name} && unzip -q -o /tmp/{name}.zip '*.png'")
    print(name, len(glob.glob(f"/tmp/blk_{name}/*.png")), "장")
P = "nkmr_fx, vivid colors, anime screencap, explosion, fire, embers, white core orange rim, layered cel shading, no outline, sharp tips, large effect, no humans, city, night, wide shot"
N = "lowres, worst quality, text, watermark"
L = "/tmp/out/nkmr_fx_color3-000003.safetensors"
for name in ("num", "clip"):
    n = len(glob.glob(f"/tmp/blk_{name}/*.png"))
    open(f'/tmp/m_{name}.txt', 'w').write("\n".join(f"{P} --n {N} --w 1344 --h 768 --s 28 --l 6 --d 1234" for _ in range(n)))
    for s in (0.4, 0.55, 0.7):
        out = f"/kaggle/working/m/{name}/s{s}"
        os.makedirs(out, exist_ok=True)
        os.system(f"cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_gen_img.py --ckpt /tmp/ilxl.safetensors --outdir {out} --from_file /tmp/m_{name}.txt "
                  f"--image_path /tmp/blk_{name} --strength {s} --sequential_file_name --images_per_prompt 1 --fp16 --no_half_vae --sdpa --sampler euler_a "
                  f"--network_module lycoris.kohya --network_weights {L} --network_mul 1.0 --network_merge > {out}/log.txt 2>&1")
        print(name, s, len(glob.glob(out + '/*.png')), 'images')


In [ ]:
# 4) 나란히 영상: 위 줄 = num(블로킹 | 0.4 | 0.55 | 0.7), 아래 줄 = clip. 24fps, 긴 쪽 기준 3번 반복 + 깜빡임 수치
import cv2, numpy as np
rows = {}
for name in ("num", "clip"):
    rows[name] = {"blocking": sorted(glob.glob(f'/tmp/blk_{name}/*.png'))}
    for s in (0.4, 0.55, 0.7):
        rows[name][f"s{s}"] = sorted(glob.glob(f'/kaggle/working/m/{name}/s{s}/*.png'))
    print(name, {k: len(v) for k, v in rows[name].items()})
def rd(p, label):
    im = cv2.resize(cv2.imread(p), (336, 192)) if p else np.zeros((192, 336, 3), np.uint8)
    cv2.putText(im, label, (6, 20), 0, 0.55, (255, 255, 255), 2)
    return im
T = max(len(rows[n]["blocking"]) for n in rows)
vw = cv2.VideoWriter('/kaggle/working/motion1.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 24, (1344, 384))
for _ in range(3):
    for t in range(T):
        lines = []
        for name in rows:
            line = []
            for k, v in rows[name].items():
                line.append(rd(v[t % len(v)] if v else None, f"{name} {k}"))
            lines.append(np.hstack(line))
        vw.write(np.vstack(lines))
vw.release()
for name in rows:
    for k, v in rows[name].items():
        a = [cv2.imread(p).astype(np.float32) for p in v]
        if len(a) > 1:
            print(name, k, '이웃 장 평균 차', round(float(np.mean([np.abs(a[i + 1] - a[i]).mean() for i in range(len(a) - 1)])), 1))
sheet = [np.hstack([cv2.resize(cv2.imread(p), (224, 128)) for p in v]) for name in rows for v in rows[name].values() if v]
wmax = max(x.shape[1] for x in sheet)
sheet = [np.pad(x, ((0, 0), (0, wmax - x.shape[1]), (0, 0))) for x in sheet]
cv2.imwrite('/kaggle/working/motion1_sheet.jpg', np.vstack(sheet), [cv2.IMWRITE_JPEG_QUALITY, 85])
!cd /kaggle/working && zip -q -r motion1_result.zip m motion1.mp4 motion1_sheet.jpg && ls -la /kaggle/working
